In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 50)

DATA_DIR = Path('dataset')
TRAIN_PATH = DATA_DIR / 'train.pq'
ITEMS_PATH = DATA_DIR / 'items.pq'
TEST_USERS_PATH = DATA_DIR / 'test_users.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'sample_submission.csv'

## 1. Загрузка данных

In [2]:
train = pd.read_parquet(TRAIN_PATH)
items = pd.read_parquet(ITEMS_PATH)
test_users = pd.read_csv(TEST_USERS_PATH) if TEST_USERS_PATH.exists() else None
sample_sub = pd.read_csv(SAMPLE_SUB_PATH) if SAMPLE_SUB_PATH.exists() else None

print('train     :', train.shape)
print('items     :', items.shape)
if test_users is not None: print('test_users:', test_users.shape)
if sample_sub is not None: print('sample_sub:', sample_sub.shape)

train     : (11373426, 6)
items     : (34323, 4)
test_users: (185282, 1)
sample_sub: (3705640, 2)


---
## 9. Моделирование

Дальше реализуем план из [`plans/HW4_recsys_plan.md`](../plans/HW4_recsys_plan.md): time-split → datamart →
baselines → ALS → EASE → DSSM (cold-friendly two-tower) → LightGBM LambdaRank.

Главная метрика — `NDCG@20`. Весь код модулирован в [`src/`](src/) и переиспользуется.

In [3]:
%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# делаем src/ импортируемым
HERE = Path('.').resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from src import data as D
from src import datamart as DM
from src import metrics as Mtx
from src import eval as Ev
from src import negatives as Neg
from src import candidates as Cand
from src import features as Feat
from src import models_combination as Comb

from src.models.baselines import GlobalTopPop, RecentTopPop, UserCategoryTopPop
from src.models.als import ALSRecommender
from src.models.ease import EASE
from src.models.dssm import DSSMRecommender, ContentVocab
from src.models.sasrec import SASRecRecommender
from src.models.ranker import LGBMRanker
print('OK, src/ imported. data dir =', D.DATA_DIR)


OK, src/ imported. data dir = /Users/vovazinev/Desktop/HSE/REC_HSE/HW4/dataset


### 9.1 Загрузка + time-based split (85/10/5)

In [4]:
train = D.load_train()
items = D.load_items()
test_users = D.load_test_users()
print('train:', train.shape, '| items:', items.shape, '| test_users:', test_users.shape)

train: (11373426, 6) | items: (34323, 4) | test_users: (185282, 1)


### 9.3 ID-encoders + общая инфра для метрик

In [4]:
user_enc, item_enc = D.fit_encoders(train, items)
print(f'users in encoder: {user_enc.n:,} | items in encoder: {item_enc.n:,}')

# глобальная популярность для novelty
item_pop = train.groupby('item_id').size().to_dict()
n_train_users = train['user_id'].nunique()
results = Ev.ResultsLog()

users in encoder: 348,111 | items in encoder: 34,323


### 9.4 M0 — Baselines (нижняя планка)

In [5]:
model_pop = GlobalTopPop().fit(train)

In [8]:
model_recent = RecentTopPop(tau_days=30).fit(train, ref_ts=train["timestamp"].max())

In [9]:
model_cat = UserCategoryTopPop(n_fav_cats=5).fit(train, items)

### 9.5 M1 — ALS implicit

In [10]:
# ALS implicit — fit-or-load из models_store
als_path = ALSRecommender.latest_path()
if als_path is not None:
    print(f'[skip fit] loading ALS from {als_path}')
    model_als = ALSRecommender.load(als_path)
else:
    # confidence = 1 + α·is_purchased + β·log(1+rating), затем масштаб alpha_conf
    model_als = ALSRecommender(
        factors=128, regularization=0.05, iterations=15,
        alpha_conf=40.0, purchase_weight=1.0, rating_weight=0.5,
    )
    model_als.fit(train, user_enc, item_enc)
    model_als.save(tag='als_f128')


  0%|          | 0/15 [00:00<?, ?it/s]

[save] ALSRecommender → /Users/vovazinev/Desktop/HSE/REC_HSE/HW4/models_store/ALSRecommender/als_f128


### 9.6 M2 — EASE (closed-form linear AE)

In [11]:
# EASE на бинарной матрице покупок. λ — главный гиперпараметр.
ease_path = EASE.latest_path()
if ease_path is not None:
    print(f'[skip fit] loading EASE from {ease_path}')
    model_ease = EASE.load(ease_path)
else:
    model_ease = EASE(lam=500.0).fit(train, user_enc, item_enc)
    model_ease.save(tag='ease_lam500')


[EASE] X: (348111, 34323), nnz=11,373,426
[EASE] solving inverse (34323, 34323)…
[EASE] B ready: (34323, 34323)
[save] EASE → /Users/vovazinev/Desktop/HSE/REC_HSE/HW4/models_store/EASE/ease_lam500


In [13]:
TEST_USER_IDS = test_users["user_id"].astype(int).tolist()
history_all = D.build_user_history(train, only_purchases=True)
K_RETR = 100
recs_pop_test = model_pop.recommend(TEST_USER_IDS, k=K_RETR, exclude=history_all)
recs_recent_test = model_recent.recommend(TEST_USER_IDS, k=K_RETR, exclude=history_all)
recs_cat_test = model_cat.recommend(TEST_USER_IDS, k=K_RETR, exclude=history_all)
recs_als_test = model_als.recommend(TEST_USER_IDS, k=K_RETR, exclude=history_all)
recs_ease_test = model_ease.recommend(TEST_USER_IDS, k=K_RETR, exclude=history_all)

blend_cand_test = Comb.build_blend_candidates({
    "pop": recs_pop_test,
    "recent": recs_recent_test,
    "cat": recs_cat_test,
    "als": recs_als_test,
    "ease": recs_ease_test,
})

blend_cand_test = Comb.add_blend_features(blend_cand_test)

uvt = blend_cand_test["user_id"].tolist()
ivt = blend_cand_test["item_id"].tolist()

blend_cand_test["score_als"] = model_als.score(uvt, ivt)
blend_cand_test["score_ease"] = model_ease.score(uvt, ivt)

blend_cand_test = Comb.add_userwise_normalized_scores(
    blend_cand_test,
    ["score_als", "score_ease"],
)


ALS.score:   0%|          | 0/94 [00:00<?, ?it/s]

EASE.score:   0%|          | 0/46 [00:00<?, ?it/s]

In [14]:
best_rrf_params = {'c': 20.0,
 'source_weights': {'pop': 0.009259781821123414,
  'recent': 0.18832197481135182,
  'cat': 0.046925269352466825,
  'als': 0.3318884026866098,
  'ease': 1.8604478598848888}}

In [15]:
final_recs = Comb.recommend_by_rrf(
    blend_cand_test,
    sources=["pop", "recent", "cat", "als", "ease"],
    c=best_rrf_params["c"],
    source_weights=best_rrf_params["source_weights"],
    user_ids=TEST_USER_IDS,
    k=20,
    exclude=history_all,
)


In [24]:
weights_v0 = {
    # rank-based
    "inv_rank_pop": 0.20,
    "inv_rank_recent": 0.30,
    "inv_rank_cat": 0.50,
    "inv_rank_als": 1.00,
    "inv_rank_ease": 0.5,

    # score-based
    "score_als_u_norm": 0.5,
    "score_ease_u_norm": 1.00,

    # source indicators
    "from_pop": 0.05,
    "from_recent": 0.05,
    "from_cat": 0.05,
    "from_als": 0.05,
    "from_ease": 0.10,
}





recs_blend_v0 = Comb.recommend_by_weights(
    blend_cand_test,
    weights_v0,
    user_ids=TEST_USER_IDS,
    k=20,
    exclude=history_all,
)


recs_blend_v0

{3: [17679,
  23777,
  610,
  9271,
  28715,
  13628,
  779,
  796,
  34003,
  12946,
  2110,
  19341,
  18327,
  18901,
  4894,
  32575,
  28869,
  18575,
  29120,
  9702],
 4: [18680,
  4941,
  18467,
  12642,
  779,
  28715,
  8790,
  27370,
  8341,
  21275,
  34277,
  1534,
  11447,
  32789,
  14743,
  7905,
  31376,
  31099,
  33834,
  9702],
 5: [33373,
  8341,
  779,
  27191,
  28715,
  32253,
  11628,
  12437,
  12378,
  31376,
  23102,
  31629,
  6261,
  14984,
  33079,
  25869,
  12642,
  5719,
  19341,
  22085],
 6: [28715,
  21252,
  33827,
  4184,
  23020,
  34277,
  15474,
  13478,
  2110,
  34003,
  7559,
  5665,
  11237,
  5520,
  19432,
  5816,
  22085,
  610,
  31099,
  24008],
 9: [28715,
  18327,
  2110,
  4941,
  19535,
  15474,
  33634,
  14765,
  22085,
  26545,
  9702,
  33827,
  34003,
  32789,
  3998,
  5520,
  17607,
  31376,
  27165,
  610],
 10: [34003,
  32081,
  5650,
  13628,
  779,
  796,
  26929,
  16520,
  12946,
  28715,
  29120,
  8142,
  29209,
  1

In [26]:
final_recs = recs_blend_v0

In [27]:
pop_fallback = model_pop.popular

for uid in TEST_USER_IDS:
    rec = final_recs.get(uid, [])
    seen = history_all.get(uid, set()) | set(rec)

    if len(rec) < 20:
        for iid in pop_fallback:
            iid = int(iid)
            if iid not in seen:
                rec.append(iid)
                seen.add(iid)
            if len(rec) == 20:
                break

    final_recs[uid] = rec[:20]


In [28]:
submission = Ev.to_submission(final_recs, TEST_USER_IDS)
print(submission.shape)
submission.to_csv("submission_blend_v0_full.csv", index=False)
submission.head()


(3705640, 2)


,user_id,item_id
0,3,17679
1,3,23777
2,3,610
3,3,9271
4,3,28715
